# Weapon Detection - YOLOv11s Training (Kaggle)

## Setup Instructions
1. Upload `weapon_detection_v2.zip` as a Kaggle Dataset first
   - Go to kaggle.com -> Datasets -> New Dataset -> upload the zip
   - Name it `weapon-detection-v2`
2. In this notebook: Add Data -> Your Datasets -> select `weapon-detection-v2`
3. Enable GPU: Settings (right sidebar) -> Accelerator -> GPU T4 x2
4. Run all cells

In [ ]:
# Cell 1: Install and check GPU
!pip install -q ultralytics>=8.3.0

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Extract dataset
# Kaggle datasets are mounted at /kaggle/input/DATASET_NAME/
# Update the path below to match your dataset name

import os
import zipfile

# Find the zip file in kaggle input
input_dir = '/kaggle/input'
zip_path = None
for root, dirs, files in os.walk(input_dir):
    for f in files:
        if f.endswith('.zip'):
            zip_path = os.path.join(root, f)
            break
    if zip_path:
        break

# Also check if dataset was uploaded unzipped (Kaggle auto-extracts)
dataset_dir = None
for root, dirs, files in os.walk(input_dir):
    if 'images' in dirs and 'labels' in dirs:
        dataset_dir = root
        break
    # Check one level deeper
    for d in dirs:
        subpath = os.path.join(root, d)
        if os.path.isdir(subpath):
            subdirs = os.listdir(subpath)
            if 'images' in subdirs and 'labels' in subdirs:
                dataset_dir = subpath
                break

work_dir = '/kaggle/working/datasets/weapon_detection_v2'

if dataset_dir:
    # Kaggle auto-extracted the zip — copy to working dir
    print(f"Found extracted dataset at: {dataset_dir}")
    if dataset_dir != work_dir:
        import shutil
        os.makedirs(os.path.dirname(work_dir), exist_ok=True)
        if not os.path.exists(work_dir):
            shutil.copytree(dataset_dir, work_dir)
        print(f"Copied to {work_dir}")
elif zip_path:
    print(f"Found zip: {zip_path}")
    print("Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/kaggle/working/datasets/')
    print("Extracted!")
else:
    print("ERROR: No dataset found!")
    print(f"Contents of {input_dir}:")
    for item in os.listdir(input_dir):
        print(f"  {item}")

# Verify
for split in ['train', 'val', 'test']:
    img_dir = f'{work_dir}/images/{split}'
    if os.path.exists(img_dir):
        count = len(os.listdir(img_dir))
        print(f"{split}: {count} images")
    else:
        print(f"{split}: NOT FOUND at {img_dir}")

In [ ]:
# Cell 3: Train YOLOv11s

dataset_yaml = """
path: /kaggle/working/datasets/weapon_detection_v2
train: images/train
val: images/val
test: images/test

nc: 4
names: ['handgun', 'long_gun', 'knife', 'explosive']
"""

with open('/kaggle/working/dataset.yaml', 'w') as f:
    f.write(dataset_yaml)

from ultralytics import YOLO

model = YOLO('yolo11s.pt')

results = model.train(
    data='/kaggle/working/dataset.yaml',
    epochs=120,
    patience=20,
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    warmup_epochs=5,
    amp=True,
    workers=2,
    # Augmentation
    mosaic=1.0,
    scale=0.5,
    degrees=10.0,
    mixup=0.1,
    fliplr=0.5,
    flipud=0.0,
    # Saving
    save=True,
    save_period=10,
    project='/kaggle/working/runs',
    name='weapon_detector_v2',
)

print(f"\nTraining complete! Best weights at: {results.save_dir}/weights/best.pt")

In [ ]:
# Cell 4: Evaluate on test set
from ultralytics import YOLO
from pathlib import Path

best_path = Path(results.save_dir) / 'weights' / 'best.pt'
model = YOLO(str(best_path))

test_results = model.val(data='/kaggle/working/dataset.yaml', split='test')

classes = ['handgun', 'long_gun', 'knife', 'explosive']
print('\n' + '=' * 60)
print('TEST SET RESULTS')
print('=' * 60)
print(f'  mAP@50:    {test_results.box.map50:.4f}')
print(f'  mAP@50-95: {test_results.box.map:.4f}')
print(f'\n{"Class":>12s} {"Precision":>10s} {"Recall":>10s} {"mAP@50":>10s}')
print('-' * 45)
for i, cls in enumerate(classes):
    if i < len(test_results.box.ap50):
        p = test_results.box.p[i] if i < len(test_results.box.p) else 0
        r = test_results.box.r[i] if i < len(test_results.box.r) else 0
        ap50 = test_results.box.ap50[i]
        print(f'  {cls:>10s} {p:>10.4f} {r:>10.4f} {ap50:>10.4f}')

# Show plots
from IPython.display import Image, display
results_dir = Path(results.save_dir)
for plot in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    plot_path = results_dir / plot
    if plot_path.exists():
        print(f'\n{plot}:')
        display(Image(filename=str(plot_path), width=800))

In [ ]:
# Cell 5: Save best.pt as notebook output
# After running, go to the notebook Output tab to download best.pt
import shutil
from pathlib import Path

best_path = Path(results.save_dir) / 'weights' / 'best.pt'
output_path = Path('/kaggle/working/best.pt')
shutil.copy2(best_path, output_path)
print(f'best.pt copied to {output_path}')
print(f'Size: {output_path.stat().st_size / 1024 / 1024:.1f} MB')
print('\nDownload from the Output tab on the right sidebar, or after saving the notebook version.')